In [1]:
import warnings
warnings.filterwarnings("ignore")

In [2]:
# совместимый версии
!pip install torchtext=='0.18.0' torch=='2.3.0' torchdata=='0.9.0' portalocker

In [3]:
import torch
import pandas as pd
import torch.nn as nn
import torch.optim as optim
from torchtext import datasets
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchtext.data import get_tokenizer
from torchtext.vocab import build_vocab_from_iterator

In [4]:
train_data = list(datasets.AG_NEWS(split='train'))
test_data = list(datasets.AG_NEWS(split='test'))

In [5]:
tokenizer = get_tokenizer('basic_english')

In [6]:
# берём только 5 строк чтобы посмотреть какой столбец первым приходит
for i in list(train_data)[:5]:
    print(i)

(3, "Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.")
(3, 'Carlyle Looks Toward Commercial Aerospace (Reuters) Reuters - Private investment firm Carlyle Group,\\which has a reputation for making well-timed and occasionally\\controversial plays in the defense industry, has quietly placed\\its bets on another part of the market.')
(3, "Oil and Economy Cloud Stocks' Outlook (Reuters) Reuters - Soaring crude prices plus worries\\about the economy and the outlook for earnings are expected to\\hang over the stock market next week during the depth of the\\summer doldrums.")
(3, 'Iraq Halts Oil Exports from Main Southern Pipeline (Reuters) Reuters - Authorities have halted oil export\\flows from the main pipeline in southern Iraq after\\intelligence showed a rebel militia could strike\\infrastructure, an oil official said on Saturday.')
(3, 'Oil prices soar to all-time record, posing new menace t

In [7]:
def text_to_tokenizer(data):
    for label, text in data:
        yield tokenizer(text)

In [8]:
vocab = build_vocab_from_iterator(text_to_tokenizer(train_data), specials=["<unk>"])

In [9]:
vocab.set_default_index(vocab["<unk>"])

In [10]:
def change_label(label): # приводим индексы классов 1,2,3,4 --> 0,1,2,3
    return label - 1

def change_text(x):
    return [vocab[i] for i in tokenizer(x)]

In [11]:
def collate_batch(batch):
    labels, texts = [], []
    for label, text in batch:
        labels.append(change_label(label))
        texts.append(torch.tensor(change_text(text), dtype=torch.int64))
    labels = torch.tensor(labels, dtype=torch.int64)
    texts = nn.utils.rnn.pad_sequence(texts, batch_first=True)
    return texts, labels

In [12]:
train_dataloader = DataLoader(list(train_data), batch_size=32, shuffle=True, collate_fn=collate_batch)
test_dataloader = DataLoader(list(test_data), batch_size=32, collate_fn=collate_batch)

In [13]:
class CheckNews(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, hidden_dim=128, output_dim=4, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.lin = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        x = self.embedding(x)
        _, (hidden, _) = self.lstm(x)
        x = self.dropout(hidden[-1])
        return self.lin(x)

In [14]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [15]:
model_CheckNews = CheckNews(len(vocab)).to(device)

In [16]:
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_CheckNews.parameters(), lr=0.001)

[transformers] Disabling PyTorch because PyTorch >= 2.4 is required but found 2.3.0
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [17]:
for epoch in range(25):
    model_CheckNews.train()
    total_loss = 0
    for texts, labels in train_dataloader:
        texts, labels = texts.to(device), labels.to(device)
        optimizer.zero_grad()
        pred = model_CheckNews(texts)
        loss = loss_fn(pred, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f'Эпоха: {epoch + 1} - Потери: {round(total_loss, 2)}')

Эпоха: 1 - Потери: 4124.34
Эпоха: 2 - Потери: 1323.72
Эпоха: 3 - Потери: 893.04
Эпоха: 4 - Потери: 658.45
Эпоха: 5 - Потери: 489.26
Эпоха: 6 - Потери: 359.3
Эпоха: 7 - Потери: 259.93
Эпоха: 8 - Потери: 190.85
Эпоха: 9 - Потери: 146.74
Эпоха: 10 - Потери: 117.17
Эпоха: 11 - Потери: 94.68
Эпоха: 12 - Потери: 75.86
Эпоха: 13 - Потери: 69.45
Эпоха: 14 - Потери: 58.81
Эпоха: 15 - Потери: 51.48
Эпоха: 16 - Потери: 47.75
Эпоха: 17 - Потери: 45.5
Эпоха: 18 - Потери: 38.53
Эпоха: 19 - Потери: 38.25
Эпоха: 20 - Потери: 31.75
Эпоха: 21 - Потери: 32.15
Эпоха: 22 - Потери: 30.88
Эпоха: 23 - Потери: 26.68
Эпоха: 24 - Потери: 26.8
Эпоха: 25 - Потери: 24.66


In [18]:
model_CheckNews.eval()
correct, total = 0, 0

with torch.no_grad():
    for x_batch, y_batch in test_dataloader:
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)

        y_pred = model_CheckNews(x_batch)
        pred = torch.argmax(y_pred, dim=1)

        correct += (pred == y_batch).sum().item()
        total += y_batch.size(0)

accuracy = correct * 100 / total
print(f'Точность предположения модели: {accuracy:.2f}%')

Точность предположения модели: 90.78%


In [19]:
def evaluate(model_CheckNews, dataloader):
    model_CheckNews.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for x_batch, y_batch in dataloader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            y_pred = model_CheckNews(x_batch)
            pred = torch.argmax(y_pred, dim=1)
            correct += (pred == y_batch).sum().item()
            total += y_batch.size(0)
    return correct / total

train_score = evaluate(model_CheckNews, train_dataloader)
test_score = evaluate(model_CheckNews, test_dataloader)

print(f'train score: {train_score:.4f}')
print(f'test score: {test_score:.4f}')

train score: 0.9989
test score: 0.9078


In [20]:
from google.colab import files

torch.save(vocab, 'vocab_CheckNews_AG_NewsClassificationDataset.pth')
torch.save(model_CheckNews.state_dict(), 'model_CheckNews_AG_NewsClassificationDataset.pth')

files.download('vocab_CheckNews_AG_NewsClassificationDataset.pth')
files.download('model_CheckNews_AG_NewsClassificationDataset.pth')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>